In [ ]:
import os
import re
import json
import time
from pathlib import Path
from dotenv import load_dotenv
from pypdf import PdfReader
import nest_asyncio

# Apply nest_asyncio to allow synchronous crew.kickoff() in Jupyter
nest_asyncio.apply()

from crewai import Agent, Task, Crew, Process, LLM
from pydantic import BaseModel, Field

load_dotenv()
print('All dependencies successfully imported and asyncio configured.')


In [ ]:
!pip install -q crewai pypdf python-dotenv pydantic

In [ ]:
!pip install -U "crewai[google-genai]"

In [ ]:
llm = LLM(
    model="gemini/gemini-3.1-flash-lite",
    api_key=os.getenv("GEMINI_API_KEY"),
    temperature=0
)


In [ ]:
VERA_SCOPE = {
    "primary_domains": [
        "Clinical Genetics",
        "Genomic Medicine",
        "Rare Diseases",
        "Neuromuscular Disorders",
        "Cardiology",
        "Clinical Pharmacology"
    ],

    "preferred_document_types": [
        "Clinical Practice Guideline",
        "Consensus Statement",
        "Clinical Protocol",
        "Systematic Review",
        "Regulatory Document",
        "Evidence-based Clinical Standard"
    ]
}

VERA_SCOPE_TEXT = json.dumps(VERA_SCOPE, indent=2)

In [ ]:
def build_document_profile(pdf_data: dict, max_abstract_chars: int = 1800) -> str:
    """
    Intelligently extracts a lightweight, highly-informative profile from the PDF:
    1. Page 1 Front Matter: Title, Authors, Journal & Abstract.
    2. High-Yield Clinical Section Snippets (scanning for Recommendations, Guidelines, Dosing, Diagnosis, Methods).
    3. Final Page Conclusion / Summary.
    Reduces token consumption by >80% while retaining all context needed for accurate classification.
    """
    pages = pdf_data["pages"]
    total_pages = len(pages)
    
    profile_parts = [
        f"FILE NAME: {pdf_data['file_name']}",
        f"TOTAL PAGES: {total_pages}",
        f"TOTAL CHARACTERS: {pdf_data['total_characters']}",
        "",
        "SMART CLINICAL DOCUMENT PROFILE & EXTRACTS:",
        "=" * 60
    ]
    
    # 1. Front Matter (Title & Abstract from Page 1)
    p1_text = pages[0]["text"].strip() if pages else ""
    profile_parts.append(f"\n--- [PAGE 1: TITLE & ABSTRACT / INTRO] ---\n{p1_text[:max_abstract_chars]}")
    
    # 2. Key Clinical Section Scanner (Hunting for recommendations, protocols, diagnoses)
    CLINICAL_KEYWORDS = [
        "recommendation", "guideline", "treatment", "dosing", "protocol",
        "diagnosis", "inclusion criteria", "monitoring", "sequencing", "conclusion"
    ]
    
    found_snippets = []
    # Scan through body pages
    for p in pages[1:-1]:
        p_num = p["page"]
        t_lower = p["text"].lower()
        for kw in CLINICAL_KEYWORDS:
            if kw in t_lower:
                idx = t_lower.find(kw)
                start = max(0, idx - 80)
                end = min(len(p["text"]), idx + 350)
                clean_snippet = p["text"][start:end].replace("\n", " ").strip()
                found_snippets.append(f"• [Page {p_num} | Section Anchor: '{kw}'] ...{clean_snippet}...")
                if len(found_snippets) >= 3:
                    break
        if len(found_snippets) >= 3:
            break
            
    if found_snippets:
        profile_parts.append("\n--- [REPRESENTATIVE CLINICAL EXCERPTS] ---")
        profile_parts.extend(found_snippets)
        
    # 3. Final Page Conclusion / Summary (if document > 2 pages)
    if total_pages > 2:
        last_page = pages[-1]
        last_text = last_page["text"].strip()
        profile_parts.append(f"\n--- [PAGE {last_page['page']}: CONCLUSION / SUMMARY] ---\n{last_text[:800]}")
        
    return "\n".join(profile_parts)


In [ ]:
def extract_pdf_text(pdf_path: str) -> dict:
    reader = PdfReader(pdf_path)

    pages = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""

        pages.append({
            "page": page_number,
            "text": text
        })

    full_text = "\n".join(
        page["text"] for page in pages
    )

    return {
        "file_name": Path(pdf_path).name,
        "total_pages": len(pages),
        "total_characters": len(full_text),
        "pages": pages,
        "full_text": full_text
    }

In [ ]:
class DocumentValidation(BaseModel):
    is_medical: bool = Field(
        description="Whether the document contains genuine medical or healthcare content."
    )

    is_clinical: bool = Field(
        description="Whether the document contains clinically relevant information."
    )

    scope_match: bool = Field(
        description="Whether the document falls within VERA's supported clinical scope."
    )

    domain: str = Field(
        description="Primary medical domain identified in the document."
    )

    subdomain: str = Field(
        description="Specific medical subdomain or specialty."
    )

    document_type: str = Field(
        description="Type of document, such as Clinical Guideline, Systematic Review, Research Paper, etc."
    )

    has_clinical_recommendations: bool = Field(
        description="Whether the document contains explicit clinical recommendations."
    )

    evidence_quality: float = Field(
        ge=0,
        le=1,
        description="Estimated evidence usefulness for VERA, from 0 to 1."
    )

    confidence: float = Field(
        ge=0,
        le=1,
        description="Confidence in the classification, from 0 to 1."
    )

    decision: str = Field(
        description="One of PASS, REVIEW, or REJECT."
    )

    reason: str = Field(
        description="Short explanation for the final decision."
    )

    warnings: list[str] = Field(
        default_factory=list,
        description="Potential issues or limitations."
    )

In [ ]:
medical_agent = Agent(
    role="Medical Content Validator",
    
    goal=(
        "Determine whether a document contains genuine medical and "
        "clinically relevant content."
    ),
    
    backstory=(
        "You are a medical information validation specialist working "
        "for VERA, a clinical evidence-grounded AI system. "
        "Your job is to distinguish genuine clinical/medical documents "
        "from unrelated, non-medical, or purely technical documents."
    ),
    
    llm=llm,
    verbose=True
)

In [ ]:
scope_agent = Agent(
    role="VERA Clinical Scope Specialist",
    
    goal=(
        "Determine whether a medical document is relevant to the "
        "clinical domains supported by VERA and whether it is suitable "
        "for inclusion in the VERA knowledge base."
    ),
    
    backstory=(
        "You are a clinical knowledge-base curator responsible for "
        "maintaining the quality and scope of VERA's medical knowledge. "
        "You must reject documents that are medical but outside the "
        "supported VERA domains."
    ),
    
    llm=llm,
    verbose=True
)

In [ ]:
final_validator = Agent(
    role="Senior Clinical Knowledge Validator",
    
    goal=(
        "Make the final decision about whether a document should be "
        "accepted into the VERA clinical knowledge base."
    ),

    backstory=(
        "You are the final quality-control authority for VERA's "
        "document ingestion pipeline. You combine medical relevance, "
        "clinical scope, document type, evidence quality, and confidence "
        "to produce a conservative PASS, REVIEW, or REJECT decision."
    ),
    
    llm=llm,
    verbose=True
)

In [ ]:
medical_task = Task(
    description="""
Analyze the provided document sample.

Determine:

1. Is the document genuinely medical?
2. Is it clinically relevant?
3. What is its primary medical domain?
4. What is its subdomain?
5. What type of medical document is it?
6. Does it contain clinical recommendations?

Do not assume that a document is medical simply because it contains
medical terminology.

DOCUMENT:

{document_profile}
""",
    
    expected_output=(
        "A structured medical relevance analysis containing "
        "medical status, clinical relevance, domain, subdomain, "
        "document type, recommendations status, and confidence."
    ),
    
    agent=medical_agent
)

In [ ]:
scope_task = Task(
    description="""
Evaluate whether the document is suitable for the VERA knowledge base.

VERA SUPPORTED SCOPE:
{vera_scope}

Review the medical analysis provided from the previous task.

Determine:
1. Whether the document matches VERA's clinical scope.
2. Whether the document type is useful for evidence-grounded RAG.
3. Whether it contains clinically useful evidence.
4. Whether it should PASS, REVIEW, or REJECT.

Be conservative. A document being medical does NOT automatically
mean that it belongs to VERA's scope.
""",
    expected_output=(
        "A structured scope and evidence-quality assessment "
        "with scope match, evidence quality, recommendation status, "
        "and a proposed decision."
    ),
    agent=scope_agent,
    context=[medical_task]
)


In [ ]:
final_task = Task(
    description="""
You are the final validator for VERA.

Review the medical analysis and scope analysis from the previous tasks.

VERA SCOPE:
{vera_scope}

Make exactly one final decision:

PASS:
The document is medically relevant, within VERA scope, and suitable
for ingestion.

REVIEW:
The document may be relevant but there is meaningful uncertainty
about scope, evidence quality, document quality, or clinical usefulness.

REJECT:
The document is non-medical, clinically irrelevant, outside VERA scope,
or unsuitable for the clinical knowledge base.

Important:
Do NOT accept a document simply because it is medical.
Return a conservative final decision and explain why.
""",
    expected_output="A final structured VERA document validation result.",
    output_pydantic=DocumentValidation,
    agent=final_validator,
    context=[medical_task, scope_task]
)


In [ ]:
crew = Crew(
    agents=[
        medical_agent,
        scope_agent,
        final_validator
    ],
    
    tasks=[
        medical_task,
        scope_task,
        final_task
    ],
    
    process=Process.sequential,
    verbose=True
)

In [ ]:
async def validate_document(pdf_path: str):
    
    print("=" * 70)
    print("VERA DOCUMENT VALIDATION")
    print("=" * 70)
    
    # 1. Extract PDF
    pdf_data = extract_pdf_text(pdf_path)
    
    print(f"\nFile: {pdf_data['file_name']}")
    print(f"Pages: {pdf_data['total_pages']}")
    print(f"Characters: {pdf_data['total_characters']}")
    
    # 2. Build profile
    document_profile = build_document_profile(pdf_data)
    
    # 3. Run Crew Asynchronously
    result = await crew.kickoff_async(
        inputs={
            "document_profile": document_profile,
            "vera_scope": VERA_SCOPE_TEXT
        }
    )
    
    print("\n" + "=" * 70)
    print("FINAL RESULT")
    print("=" * 70)
    
    if hasattr(result, "pydantic") and result.pydantic:
        validation = result.pydantic
        print(validation.model_dump_json(indent=2))
        return validation
    
    print(result.raw)
    return result


In [ ]:
from pathlib import Path

pdf_path = Path(
    r"D:\AI Hackathon\New data\data\raw_pdfs\GenomeResearch_2024_LongRead_Chromosomal_Rearrangements.pdf"
)

print("Exists:", pdf_path.exists())
print("Path:", pdf_path.resolve())

In [ ]:
from pathlib import Path

pdf_path = Path(
    r"D:\pdfs\Complete ROS 2 MCQ Quiz.pdf"
)

result = await validate_document(str(pdf_path))